In [8]:
# NGAI - Next Gen Artificial Intelligence | GPU Training Notebook
# Trains dense, MoE, and novel-routing NGAI models on Colab GPU.
# Before running: Runtime > Change runtime type > GPU

import os, shutil, subprocess
repo_dir = '/content/NGAI'
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
subprocess.run(['git', 'clone', 'https://github.com/iceyxsm/NGAI.git'], cwd='/content', check=True)
os.chdir(repo_dir)
!pip install -e . -q
print('\n✅ NGAI installed successfully')

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 1.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 7.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.6 MB/s eta 0:00:0000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 128.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.5 MB/s eta

In [12]:
# Cell 2: Verify GPU availability
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  No GPU detected! Go to Runtime > Change runtime type > GPU')

Device: cuda
GPU:    Tesla T4


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [11]:
# Cell 3: Download TinyShakespeare dataset
!mkdir -p data
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O data/tinyshakespeare.txt
import os
size = os.path.getsize('data/tinyshakespeare.txt')
print(f'✅ TinyShakespeare downloaded: {size:,} bytes ({size/1024:.0f} KB)')

✅ TinyShakespeare downloaded: 1,115,394 bytes (1089 KB)


In [ ]:
# Cell 4: Train Dense model on GPU
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from ngai.data.text_dataset import CharDataset
from ngai.models.ngai_lm import NGAILanguageModel
from ngai.utils.seed import set_seed

# --- Hyperparameters ---
DIM = 256
N_LAYERS = 8
SEQ_LEN = 128
BATCH_SIZE = 128
LR = 3e-4
STEPS = 2000
LOG_EVERY = 200

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Data ---
set_seed(42)
dataset = CharDataset.from_file('data/tinyshakespeare.txt', SEQ_LEN)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
vocab = dataset.vocab_size
print(f'Vocab: {vocab}, Sequences: {len(dataset):,}')

# --- Dense model ---
dense_model = NGAILanguageModel(vocab, DIM, N_LAYERS).to(DEVICE)
dense_params = dense_model.count_parameters()
print(f'Dense model params: {dense_params:,}')
print(f'Training for {STEPS} steps on {DEVICE}...\n')

optimizer = torch.optim.AdamW(dense_model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
dense_model.train()

loader_iter = iter(loader)
dense_losses = []
t0 = time.perf_counter()

for step in range(STEPS):
    try:
        x, y = next(loader_iter)
    except StopIteration:
        loader_iter = iter(loader)
        x, y = next(loader_iter)

    x, y = x.to(DEVICE), y.to(DEVICE)
    logits, _ = dense_model(x)
    loss = criterion(logits.view(-1, vocab), y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(dense_model.parameters(), max_norm=1.0)
    optimizer.step()

    dense_losses.append(loss.item())
    if (step + 1) % LOG_EVERY == 0:
        avg = sum(dense_losses[-LOG_EVERY:]) / LOG_EVERY
        elapsed = time.perf_counter() - t0
        sps = (step + 1) / elapsed
        print(f'  step {step+1:>5} | loss {avg:.4f} | {sps:.1f} steps/sec')

dense_time = time.perf_counter() - t0
dense_final = sum(dense_losses[-50:]) / 50
print(f'\n✅ Dense training done: {dense_final:.4f} final loss, {dense_time:.1f}s total')

In [ ]:
# Cell 5: Train MoE model on GPU
from ngai.models.ngai_moe_lm import NGAIMoELanguageModel

# --- MoE config ---
N_SHARED = 1
N_ROUTED = 8
TOP_K = 2
MOE_EXPAND = 2

set_seed(42)
moe_model = NGAIMoELanguageModel(
    vocab, DIM, N_LAYERS, N_SHARED, N_ROUTED, TOP_K, MOE_EXPAND
).to(DEVICE)
moe_total = moe_model.count_parameters()
moe_active = moe_model.count_active_parameters()
print(f'MoE total params:  {moe_total:,}')
print(f'MoE active/token:  {moe_active:,}')
print(f'Sparsity:          {1 - moe_active/moe_total:.1%}')
print(f'Training for {STEPS} steps on {DEVICE}...\n')

optimizer_moe = torch.optim.AdamW(moe_model.parameters(), lr=LR)
moe_model.train()

loader_iter = iter(loader)
moe_losses = []
t0 = time.perf_counter()

for step in range(STEPS):
    try:
        x, y = next(loader_iter)
    except StopIteration:
        loader_iter = iter(loader)
        x, y = next(loader_iter)

    x, y = x.to(DEVICE), y.to(DEVICE)
    logits, _, balance_loss = moe_model(x)
    ce_loss = criterion(logits.view(-1, vocab), y.view(-1))
    loss = ce_loss + balance_loss

    optimizer_moe.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(moe_model.parameters(), max_norm=1.0)
    optimizer_moe.step()

    moe_losses.append(ce_loss.item())
    if (step + 1) % LOG_EVERY == 0:
        avg = sum(moe_losses[-LOG_EVERY:]) / LOG_EVERY
        elapsed = time.perf_counter() - t0
        sps = (step + 1) / elapsed
        print(f'  step {step+1:>5} | loss {avg:.4f} | {sps:.1f} steps/sec')

moe_time = time.perf_counter() - t0
moe_final = sum(moe_losses[-50:]) / 50
print(f'\n✅ MoE training done: {moe_final:.4f} final loss, {moe_time:.1f}s total')

In [ ]:
# Cell 6: Generate samples from both models

@torch.no_grad()
def generate(model, dataset, prompt, max_tokens=300, temperature=0.8, is_moe=False):
    """Generate text autoregressively."""
    model.eval()
    indices = [dataset.char_to_idx.get(c, 0) for c in prompt]
    tokens = torch.tensor([indices], dtype=torch.long).to(DEVICE)

    if is_moe:
        logits, states, _ = model(tokens)
    else:
        logits, states = model(tokens)

    generated = list(prompt)
    for _ in range(max_tokens):
        next_logit = logits[:, -1, :]
        probs = torch.softmax(next_logit / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated.append(dataset.idx_to_char[next_token.item()])
        if is_moe:
            logits, states, _ = model(next_token, states)
        else:
            logits, states = model(next_token, states)

    return ''.join(generated)

print('=' * 60)
print('  DENSE MODEL SAMPLE')
print('=' * 60)
print(generate(dense_model, dataset, 'ROMEO:\n', is_moe=False))

print()
print('=' * 60)
print('  MOE MODEL SAMPLE')
print('=' * 60)
print(generate(moe_model, dataset, 'ROMEO:\n', is_moe=True))

In [ ]:
# Cell 7: Novel Routing Comparison
# Trains 3 MoE variants: Standard, State-Aware, Adaptive
from ngai.core.novel_routing import AdaptiveRouter, StateAwareRouter

ROUTING_STEPS = 2000

def make_moe_model(vocab_size, router_type):
    """Create an MoE model with the specified router type."""
    model = NGAIMoELanguageModel(
        vocab_size, DIM, N_LAYERS, N_SHARED, N_ROUTED, TOP_K, MOE_EXPAND
    )
    if router_type == 'state_aware':
        for block in model.blocks:
            block.channel_mixer.router = StateAwareRouter(DIM, N_ROUTED, TOP_K)
    elif router_type == 'adaptive':
        for block in model.blocks:
            block.channel_mixer.router = AdaptiveRouter(DIM, N_ROUTED, TOP_K)
    return model.to(DEVICE)

def train_variant(name, model, loader, steps):
    """Train one MoE variant and return results dict."""
    print(f'\n  Training: {name} ({model.count_parameters():,} params)')
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()
    model.train()
    loader_iter = iter(loader)
    losses = []
    t0 = time.perf_counter()

    for step in range(steps):
        try:
            x, y = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x, y = next(loader_iter)

        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, _, balance_loss = model(x)
        ce_loss = crit(logits.view(-1, vocab), y.view(-1))
        loss = ce_loss + balance_loss

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        losses.append(ce_loss.item())
        if (step + 1) % LOG_EVERY == 0:
            avg = sum(losses[-LOG_EVERY:]) / LOG_EVERY
            elapsed = time.perf_counter() - t0
            sps = (step + 1) / elapsed
            print(f'    step {step+1:>5} | loss {avg:.4f} | {sps:.1f} steps/sec')

    elapsed = time.perf_counter() - t0
    final = sum(losses[-50:]) / 50 if len(losses) >= 50 else sum(losses) / len(losses)
    return {'name': name, 'params': model.count_parameters(),
            'final_loss': final, 'time': elapsed, 'sps': steps / elapsed}

# --- Train all 3 variants ---
variants = [
    ('Standard MoE',       'standard'),
    ('State-Aware Router', 'state_aware'),
    ('Adaptive Router',    'adaptive'),
]

results = []
for name, rtype in variants:
    set_seed(42)
    model = make_moe_model(vocab, rtype)
    result = train_variant(name, model, loader, ROUTING_STEPS)
    results.append(result)
    del model
    torch.cuda.empty_cache()

# --- Print comparison table ---
print(f'\n{"=" * 65}')
print('  NOVEL ROUTING COMPARISON')
print(f'{"=" * 65}')
print(f'  {"Model":<25} {"Params":>10} {"Loss":>10} {"Steps/s":>10}')
print(f'  {"-"*25} {"-"*10} {"-"*10} {"-"*10}')
for r in results:
    print(f'  {r["name"]:<25} {r["params"]:>10,} {r["final_loss"]:>10.4f} {r["sps"]:>10.1f}')
print(f'{"=" * 65}')

best = min(results, key=lambda r: r['final_loss'])
print(f'\n  \U0001f3c6 Best: {best["name"]} (loss {best["final_loss"]:.4f}, {best["sps"]:.1f} steps/sec)')